In [0]:
-- Create the test_gold schema (the only schema added by this notebook)
CREATE SCHEMA IF NOT EXISTS public_health.test_gold;


In [0]:
-- Dimension view: geography at province and region level
-- Drop existing table (from gold pipeline) so we can create a view with the same name
DROP TABLE IF EXISTS public_health.test_gold.dim_geography;
CREATE OR REPLACE VIEW public_health.test_gold.dim_geography AS
SELECT
    CAST(g.psgc_code AS BIGINT) AS geography_key,
    g.psgc_code,
    g.area_name,
    g.geographic_level,
    g.region_code,
    g.province_code,
    r.area_name AS region_name
FROM public_health.silver.geography_silver g
LEFT JOIN public_health.silver.geography_silver r
    ON g.region_code = r.region_code
    AND r.geographic_level = 'Reg'
WHERE g.geographic_level IN ('Prov', 'Reg');


In [0]:
-- Dimension view: geography at province and region level
CREATE OR REPLACE VIEW public_health.test_gold.dim_geography AS
SELECT
    CAST(g.psgc_code AS BIGINT) AS geography_key,
    g.psgc_code,
    g.area_name,
    g.geographic_level,
    g.region_code,
    g.province_code,
    r.area_name AS region_name
FROM public_health.silver.geography_silver g
LEFT JOIN public_health.silver.geography_silver r
    ON g.region_code = r.region_code
    AND r.geographic_level = 'Reg'
WHERE g.geographic_level IN ('Prov', 'Reg');


In [0]:
-- Dimension view: disease indicators from IHME GBD (selected indicator set)
CREATE OR REPLACE VIEW public_health.test_gold.dim_disease AS
SELECT
    ROW_NUMBER() OVER (ORDER BY cause_name) AS disease_key,
    cause_name AS indicator_name,
    'IHME_GBD' AS source,
    'Deaths' AS measure_type,
    'Number' AS metric_type
FROM (
    SELECT DISTINCT cause_name
    FROM public_health.bronze.diseases_ihme_gbd_bronze
    WHERE measure_name = 'Deaths'
);


In [0]:
-- Fact table: area burden priority
-- Grain: one record per (planning_unit_key, burden_indicator_key, weight_set, eligibility_config)
CREATE TABLE IF NOT EXISTS public_health.test_gold.fact_area_burden_priority (
    -- Grain Composite Primary Keys
    planning_unit_key        BIGINT        NOT NULL COMMENT 'Province-level planning unit key (PSGC code as BIGINT from geography_silver)',
    burden_indicator_key     BIGINT        NOT NULL COMMENT 'Disease indicator key from dim_disease',
    weight_set               STRING        NOT NULL COMMENT 'Weight parameter set identifier',
    eligibility_config       STRING        NOT NULL COMMENT 'Eligibility rule configuration',

    -- Profile & Burden Metrics
    service_profile_score    DECIMAL(18,4)          COMMENT 'Normalized service availability score (0-1) based on facility counts',
    burden_value             DECIMAL(18,4)          COMMENT 'Disease burden value (death count from IHME GBD 2023)',
    applied_weight           DECIMAL(10,6)          COMMENT 'Weight value applied from weight parameter set',

    -- Derived Gold Business Logic
    is_eligible              BOOLEAN       NOT NULL COMMENT 'Eligibility status based on eligibility_config rules',
    weighted_burden_score    DECIMAL(18,6)          COMMENT 'Priority metric: burden_value * applied_weight',
    final_priority_rank      INT                    COMMENT 'Rank within planning unit derived from weighted score',

    -- System & Metadata Audit Fields
    etl_inserted_at          TIMESTAMP     NOT NULL DEFAULT CURRENT_TIMESTAMP(),
    etl_updated_at           TIMESTAMP     NOT NULL DEFAULT CURRENT_TIMESTAMP(),
    batch_id                 STRING                 COMMENT 'Ingestion pipeline execution run ID',

    CONSTRAINT pk_fact_area_burden_priority
        PRIMARY KEY (planning_unit_key, burden_indicator_key, weight_set, eligibility_config)
)
USING DELTA
CLUSTER BY (planning_unit_key, burden_indicator_key)
COMMENT 'Gold layer table representing prioritized area burden profiles per weight set and eligibility rule.'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true',
    'delta.feature.allowColumnDefaults' = 'supported'
);


In [0]:
-- Merge burden + service profile data into the fact table
WITH province_geo AS (
    SELECT DISTINCT
        CAST(g.psgc_code AS BIGINT) AS planning_unit_key,
        g.area_name AS planning_unit_name
    FROM public_health.silver.geography_silver g
    WHERE g.geographic_level = 'Prov'
),
service_profile AS (
    SELECT
        CAST(CONCAT(SUBSTRING(f.psgc_code, 1, 5), '00000') AS BIGINT) AS planning_unit_key,
        COUNT(CASE WHEN f.facility_category = 'hospital' THEN 1 END) AS hospital_count,
        COUNT(CASE WHEN f.facility_category = 'primary_care' THEN 1 END) AS primary_care_count,
        SUM(CASE WHEN f.facility_category = 'hospital' THEN COALESCE(f.analysis_beds, 0) ELSE 0 END) AS total_beds
    FROM (
        SELECT Province_PSGC AS psgc_code, 'hospital' AS facility_category,
               TRY_CAST(REPLACE(Bed_Capacity, ',', '') AS INT) AS analysis_beds
        FROM public_health.bronze.facilities_nhfr_bronze
        WHERE Province_PSGC IS NOT NULL

        UNION ALL

        SELECT g.psgc_code, 'primary_care' AS facility_category,
               CAST(NULL AS INT) AS analysis_beds
        FROM public_health.bronze.facilities_hfsrb_primary_care_bronze pc
        INNER JOIN public_health.silver.geography_silver g
            ON lower(pc.province) = lower(g.area_name) AND g.geographic_level = 'Prov'
        WHERE pc.province IS NOT NULL
    ) f
    WHERE f.psgc_code IS NOT NULL
    GROUP BY SUBSTRING(f.psgc_code, 1, 5)
),
burden_data AS (
    SELECT
        d.disease_key AS burden_indicator_key,
        lower(i.location_name) AS location_name_lower,
        SUM(TRY_CAST(i.val AS DECIMAL(18,4))) AS burden_value
    FROM public_health.bronze.diseases_ihme_gbd_bronze i
    INNER JOIN public_health.test_gold.dim_disease d
        ON i.cause_name = d.indicator_name
    WHERE i.measure_name = 'Deaths'
      AND i.year = '2023'
      AND i.sex_name = 'Both'
      AND i.age_name = 'All ages'
    GROUP BY d.disease_key, i.location_name
),
weight_params AS (
    SELECT 'DEFAULT' AS weight_set, 'STANDARD' AS eligibility_config, CAST(1.0 AS DECIMAL(10,6)) AS applied_weight
    UNION ALL
    SELECT 'BURDEN_WEIGHTED', 'STRICT', CAST(1.5 AS DECIMAL(10,6))
),
area_burden AS (
    SELECT
        pg.planning_unit_key,
        bd.burden_indicator_key,
        COALESCE(sp.hospital_count, 0) AS hospital_count,
        COALESCE(sp.primary_care_count, 0) AS primary_care_count,
        bd.burden_value
    FROM province_geo pg
    INNER JOIN burden_data bd ON lower(pg.planning_unit_name) = bd.location_name_lower
    LEFT JOIN service_profile sp ON pg.planning_unit_key = sp.planning_unit_key
),
staged_gold AS (
    SELECT
        ab.planning_unit_key,
        ab.burden_indicator_key,
        pwp.weight_set,
        pwp.eligibility_config,
        CASE
            WHEN (ab.primary_care_count + ab.hospital_count) = 0 THEN CAST(0.0 AS DECIMAL(18,4))
            ELSE LEAST(CAST(ab.primary_care_count + ab.hospital_count AS DECIMAL(18,4)) / 50.0, 1.0)
        END AS service_profile_score,
        ab.burden_value,
        pwp.applied_weight,
        CASE
            WHEN pwp.eligibility_config = 'STRICT' AND (ab.primary_care_count + ab.hospital_count) = 0 THEN FALSE
            ELSE TRUE
        END AS is_eligible,
        (ab.burden_value * pwp.applied_weight) AS weighted_burden_score,
        CAST(NULL AS INT) AS final_priority_rank,
        '' AS batch_id
    FROM area_burden ab
    CROSS JOIN weight_params pwp
)
MERGE INTO public_health.test_gold.fact_area_burden_priority AS target
USING staged_gold AS source
ON  target.planning_unit_key    = source.planning_unit_key
AND target.burden_indicator_key = source.burden_indicator_key
AND target.weight_set           = source.weight_set
AND target.eligibility_config   = source.eligibility_config
WHEN MATCHED THEN UPDATE SET
    target.service_profile_score = source.service_profile_score,
    target.burden_value          = source.burden_value,
    target.applied_weight        = source.applied_weight,
    target.is_eligible           = source.is_eligible,
    target.weighted_burden_score = source.weighted_burden_score,
    target.batch_id              = source.batch_id,
    target.etl_updated_at        = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT (
    planning_unit_key, burden_indicator_key, weight_set, eligibility_config,
    service_profile_score, burden_value, applied_weight,
    is_eligible, weighted_burden_score, batch_id,
    etl_inserted_at, etl_updated_at
) VALUES (
    source.planning_unit_key, source.burden_indicator_key, source.weight_set, source.eligibility_config,
    source.service_profile_score, source.burden_value, source.applied_weight,
    source.is_eligible, source.weighted_burden_score, source.batch_id,
    CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()
);


In [0]:
-- Mart view: gap-type flags per planning unit and burden indicator
-- Answers: Which areas combine high disease burden with limited primary care or hospital
-- services, low public workforce supply or socioeconomic disadvantage, and what type of
-- gap does each area have?
CREATE OR REPLACE VIEW public_health.test_gold.mart_area_priority AS
WITH area_facility_hfsrb AS (
    -- HFSRB licensed facility counts per province
    SELECT
        CAST(CONCAT(SUBSTRING(f.psgc_code, 1, 5), '00000') AS BIGINT) AS planning_unit_key,
        COUNT(CASE WHEN f.facility_category = 'primary_care' THEN 1 END) AS hfsrb_primary_care_count,
        COUNT(CASE WHEN f.facility_category = 'hospital' THEN 1 END) AS hfsrb_hospital_count
    FROM (
        SELECT g.psgc_code, 'hospital' AS facility_category
        FROM public_health.bronze.facilities_hfsrb_hospitals_bronze h
        INNER JOIN public_health.silver.geography_silver g
            ON lower(h.province) = lower(g.area_name) AND g.geographic_level = 'Prov'
        WHERE h.province IS NOT NULL

        UNION ALL

        SELECT g.psgc_code, 'primary_care' AS facility_category
        FROM public_health.bronze.facilities_hfsrb_primary_care_bronze pc
        INNER JOIN public_health.silver.geography_silver g
            ON lower(pc.province) = lower(g.area_name) AND g.geographic_level = 'Prov'
        WHERE pc.province IS NOT NULL
    ) f
    WHERE f.psgc_code IS NOT NULL
    GROUP BY SUBSTRING(f.psgc_code, 1, 5)
),
area_workforce AS (
    -- FHSIS health workforce per province
    SELECT
        CAST(w.psgc_code AS BIGINT) AS planning_unit_key,
        MAX(TRY_CAST(w.population AS BIGINT)) AS population,
        MAX(TRY_CAST(w.doctors_total AS INT)) AS doctors_total,
        MAX(TRY_CAST(w.nurses_total AS INT)) AS nurses_total,
        MAX(TRY_CAST(w.midwives_total AS INT)) AS midwives_total
    FROM public_health.bronze.workforce_fhsis_health_workers_bronze w
    WHERE w.psgc_code IS NOT NULL
      AND w.psgc_code IN (
          SELECT psgc_code FROM public_health.silver.geography_silver WHERE geographic_level = 'Prov'
      )
    GROUP BY w.psgc_code
),
area_poverty AS (
    -- Poverty incidence per province (2023)
    SELECT
        CAST(g.psgc_code AS BIGINT) AS planning_unit_key,
        MAX(TRY_CAST(p.incidence_2023 AS DECIMAL(10,2))) AS poverty_incidence
    FROM public_health.bronze.socioeconomic_psa_poverty_bronze p
    INNER JOIN public_health.silver.geography_silver g
        ON lower(p.area_name) = lower(g.area_name) AND g.geographic_level = 'Prov'
    WHERE p._source_sheet = 'tab2'
      AND p.incidence_2023 IS NOT NULL
    GROUP BY g.psgc_code
),
area_fhsis_primary_care AS (
    -- FHSIS reported primary care counts per province
    SELECT
        CAST(g.psgc_code AS BIGINT) AS planning_unit_key,
        MAX(TRY_CAST(f.health_centers_total AS INT)) AS fhsis_primary_care_count
    FROM public_health.bronze.facilities_fhsis_primary_care_bronze f
    INNER JOIN public_health.silver.geography_silver g
        ON lower(f.area_name) = lower(g.area_name) AND g.geographic_level = 'Prov'
    WHERE f.health_centers_total IS NOT NULL
    GROUP BY g.psgc_code
),
gap_base AS (
    SELECT
        f.planning_unit_key,
        dg.area_name AS planning_unit_name,
        dg.region_name,
        f.burden_indicator_key,
        dd.indicator_name AS burden_indicator_name,
        f.burden_value,
        f.service_profile_score,
        f.weighted_burden_score,
        COALESCE(aw.population, 0) AS population,
        COALESCE(afh.hfsrb_primary_care_count, 0) AS hfsrb_pc_count,
        COALESCE(afh.hfsrb_hospital_count, 0) AS hfsrb_hosp_count,
        COALESCE(afc.fhsis_primary_care_count, 0) AS fhsis_pc_count,
        COALESCE(aw.doctors_total, 0) AS doctors_total,
        COALESCE(ap.poverty_incidence, 0) AS poverty_incidence
    FROM public_health.test_gold.fact_area_burden_priority f
    INNER JOIN public_health.test_gold.dim_geography dg
        ON f.planning_unit_key = dg.geography_key AND dg.geographic_level = 'Prov'
    INNER JOIN public_health.test_gold.dim_disease dd
        ON f.burden_indicator_key = dd.disease_key
    LEFT JOIN area_facility_hfsrb afh ON f.planning_unit_key = afh.planning_unit_key
    LEFT JOIN area_workforce aw       ON f.planning_unit_key = aw.planning_unit_key
    LEFT JOIN area_poverty ap         ON f.planning_unit_key = ap.planning_unit_key
    LEFT JOIN area_fhsis_primary_care afc ON f.planning_unit_key = afc.planning_unit_key
    WHERE f.weight_set = 'DEFAULT' AND f.eligibility_config = 'STANDARD'
),
gap_computed AS (
    SELECT
        *,
        -- HFSRB primary care gap: < 0.5 facilities per 10k population
        CASE
            WHEN population = 0 THEN FALSE
            WHEN hfsrb_pc_count = 0 THEN TRUE
            WHEN hfsrb_pc_count / NULLIF(population / 10000.0, 0) < 0.5 THEN TRUE
            ELSE FALSE
        END AS hfsrb_pc_gap,
        -- FHSIS primary care gap: < 0.5 facilities per 10k population
        CASE
            WHEN population = 0 THEN FALSE
            WHEN fhsis_pc_count = 0 THEN TRUE
            WHEN fhsis_pc_count / NULLIF(population / 10000.0, 0) < 0.5 THEN TRUE
            ELSE FALSE
        END AS fhsis_pc_gap
    FROM gap_base
),
gap_flags AS (
    SELECT
        *,
        -- Primary care gap (either source)
        CASE WHEN hfsrb_pc_gap OR fhsis_pc_gap THEN TRUE ELSE FALSE END AS has_primary_care_gap,
        -- Primary care gap basis
        CASE
            WHEN hfsrb_pc_gap AND fhsis_pc_gap THEN 'Both'
            WHEN hfsrb_pc_gap THEN 'HFSRB licensed list'
            WHEN fhsis_pc_gap THEN 'FHSIS reported counts'
            ELSE NULL
        END AS primary_care_gap_basis,
        -- Hospital gap: < 1 hospital per 100k population
        CASE
            WHEN population = 0 THEN FALSE
            WHEN hfsrb_hosp_count = 0 THEN TRUE
            WHEN hfsrb_hosp_count / NULLIF(population / 100000.0, 0) < 1.0 THEN TRUE
            ELSE FALSE
        END AS has_hospital_gap,
        -- Workforce gap: < 1 doctor per 10k population
        CASE
            WHEN population = 0 THEN FALSE
            WHEN doctors_total = 0 THEN TRUE
            WHEN doctors_total / NULLIF(population / 10000.0, 0) < 1.0 THEN TRUE
            ELSE FALSE
        END AS has_workforce_gap,
        -- Socioeconomic flag: poverty incidence > 20%
        CASE
            WHEN poverty_incidence > 20.0 THEN TRUE
            ELSE FALSE
        END AS has_socioeconomic_flag
    FROM gap_computed
)
SELECT
    planning_unit_key,
    planning_unit_name,
    region_name,
    burden_indicator_key,
    burden_indicator_name,
    burden_value,
    service_profile_score,
    weighted_burden_score,
    ROW_NUMBER() OVER (
        PARTITION BY planning_unit_key
        ORDER BY weighted_burden_score DESC
    ) AS final_priority_rank,
    has_primary_care_gap,
    primary_care_gap_basis,
    has_hospital_gap,
    has_workforce_gap,
    has_socioeconomic_flag,
    COALESCE(
        CONCAT_WS('+',
            CASE WHEN has_primary_care_gap THEN 'primary_care' ELSE NULL END,
            CASE WHEN has_hospital_gap THEN 'hospital' ELSE NULL END,
            CASE WHEN has_workforce_gap THEN 'workforce' ELSE NULL END,
            CASE WHEN has_socioeconomic_flag THEN 'socioeconomic' ELSE NULL END
        ),
        'none'
    ) AS gap_profile,
    (CASE WHEN has_primary_care_gap THEN 1 ELSE 0 END
   + CASE WHEN has_hospital_gap THEN 1 ELSE 0 END
   + CASE WHEN has_workforce_gap THEN 1 ELSE 0 END
   + CASE WHEN has_socioeconomic_flag THEN 1 ELSE 0 END) AS flags_count,
    'primary_care,hospital,workforce,socioeconomic' AS indicator_set,
    'NULL treated as no gap; zero facilities treated as gap' AS missing_value_rule,
    'province_total_2021' AS ranking_population
FROM gap_flags;


In [0]:
-- Preview: areas with the most gap types, highest burden first
SELECT
    planning_unit_name,
    region_name,
    burden_indicator_name,
    burden_value,
    has_primary_care_gap,
    primary_care_gap_basis,
    has_hospital_gap,
    has_workforce_gap,
    has_socioeconomic_flag,
    gap_profile,
    flags_count
FROM public_health.test_gold.mart_area_priority
WHERE flags_count > 0
ORDER BY flags_count DESC, burden_value DESC
LIMIT 20;
